# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# Import the necessary libs
import os
from datetime import datetime
from typing import List, Dict, Optional, TypedDict

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool, Tool
from lib.parsers import PydanticOutputParser

In [3]:
# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# Reconnect to the persistent vector DB built in Part 1.
# IMPORTANT: load it with the SAME embedding function used to create it (OpenAI).
# Otherwise Chroma attaches its default 384-dim function and queries fail against
# the stored 1536-dim OpenAI vectors.
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY")
)
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)


@tool
def retrieve_game(query: str) -> List[Dict]:
    """Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. 

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
        include=["documents", "metadatas", "distances"],
    )

    games = []
    for metadata in results["metadatas"][0]:
        games.append({
            "Platform": metadata["Platform"],
            "Name": metadata["Name"],
            "YearOfRelease": metadata["YearOfRelease"],
            "Description": metadata["Description"],
        })
    return games

#### Evaluate Retrieval Tool

In [5]:
# EvaluationReport is the structured output the judge LLM must return.
# (It is not part of lib/, so we define it here.)
class EvaluationReport(BaseModel):
    useful: bool = Field(
        description="Whether the retrieved documents are enough to answer the question"
    )
    description: str = Field(
        description="Detailed explanation of why the documents are or aren't enough"
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict]) -> EvaluationReport:
    """Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
    args: 
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini", temperature=0.0)

    messages = [
        SystemMessage(content=(
            "Your task is to evaluate if the documents are enough to respond the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not."
        )),
        UserMessage(content=(
            f"# Question:\n{question}\n\n"
            f"# Retrieved Documents:\n{retrieved_docs}"
        )),
    ]

    ai_message = llm.invoke(input=messages, response_format=EvaluationReport)
    evaluation = PydanticOutputParser(model_class=EvaluationReport).parse(ai_message)
    return evaluation

#### Game Web Search Tool

In [6]:
@tool
def game_web_search(question: str) -> Dict:
    """Web search: Looks up information about the game industry on the web.
    Use this as a fallback when the internal vector DB does not have a good answer.
    args:
    - question: a question about game industry. 
    """
    tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False,
    )

    return {
        "answer": response.get("answer"),
        "results": response.get("results"),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question,
        },
    }

### Agent

In [7]:
# Create your Agent.
# The Agent class is already built on top of StateMachine
# (message_prep -> llm_processor -> tool_executor -> ... -> termination),
# so we just equip it with a model, a good set of instructions, and the three tools.
INSTRUCTIONS = """You are UdaPlay, an AI research assistant for the video game industry.

You answer questions about video games (release dates, platforms, genres, publishers, etc.).
For every question, follow this strategy:

1. Call `retrieve_game` to search your internal knowledge base (the vector DB).
2. Call `evaluate_retrieval` with the user's question and the retrieved documents to
   judge whether they are enough to answer.
3. If the evaluation says the documents are NOT useful, call `game_web_search` to look
   the answer up on the web.
4. Give a clear, concise final answer based on the best evidence available.
   - If you used the web, mention that the information came from a web search.
   - If you still can't find the answer, say so honestly instead of guessing.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)


def get_final_answer(run):
    """Return the agent's final natural-language answer from a Run object."""
    messages = run.get_final_state()["messages"]
    for message in reversed(messages):
        if isinstance(message, AIMessage) and message.content:
            return message.content
    return None

In [8]:
# Invoke your agent
questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]

for i, question in enumerate(questions, start=1):
    print("=" * 80)
    print(f"Q{i}: {question}\n")
    # Use a fresh session per question so they stay independent
    run = agent.invoke(question, session_id=f"question-{i}")
    print(f"\nA{i}: {get_final_answer(run)}\n")

Q1: When was Pokémon Gold and Silver released?

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

A1: Pokémon Gold and Silver were released in 1999 for the Game Boy Color.

Q2: Which one was the first 3D platformer Mario game?

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

A2: The first 3D platformer Mario game is **Super Mario 64**, which was released for the Nintendo 64 in 1996. This game was 

### (Optional) Advanced

In [9]:
# ADVANCED 1 — Update your agent with long-term memory.
# Same pattern as Module 09: wrap LongTermMemory in tools via factory closures
# (so owner/namespace are fixed), then add them to the agent's toolset.
from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager

vector_store_manager = VectorStoreManager(os.getenv("OPENAI_API_KEY"))
long_term_memory = LongTermMemory(vector_store_manager)


def build_register_memory_tool(ltm: LongTermMemory, owner: str, namespace: str = "udaplay") -> Tool:
    def register_memory(content: str) -> str:
        """Store a useful fact learned during the conversation for future use.
        args:
        - content: the fact to remember (e.g. "Mortal Kombat X was not released for PS5")
        """
        ltm.register(MemoryFragment(content=content, owner=owner, namespace=namespace))
        return "Stored in long-term memory."
    return Tool(func=register_memory)


def build_search_memory_tool(ltm: LongTermMemory, owner: str, namespace: str = "udaplay") -> Tool:
    def search_memory(query: str) -> str:
        """Search your long-term memory for previously stored facts.
        args:
        - query: what to look up in memory
        """
        result = ltm.search(query_text=query, owner=owner, namespace=namespace)
        return str([fragment.content for fragment in result.fragments])
    return Tool(func=search_memory)


OWNER = "udaplay_user"
memory_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS + (
        "\n\nYou also have a long-term memory. You may use `search_memory` to recall "
        "useful facts from earlier conversations, and after finding a good answer use "
        "`register_memory` to store the key fact for next time."
    ),
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search,
        build_register_memory_tool(long_term_memory, owner=OWNER),
        build_search_memory_tool(long_term_memory, owner=OWNER),
    ],
    temperature=0.0,
)

# Demo: store a fact in one session, recall it in a different session
run = memory_agent.invoke(
    "Was Mortal Kombat X released for PlayStation 5? Please remember the answer.",
    session_id="mem-write",
)
print("First answer :", get_final_answer(run))

run = memory_agent.invoke(
    "From what you remember, was Mortal Kombat X on PS5?",
    session_id="mem-read",
)
print("Recalled     :", get_final_answer(run))

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
First answer : Mortal Kombat X was not released for PlayStation 5. It was originally released for PlayStation 4, Xbox One, and PC in 2015. An upgraded version, Mortal Kombat XL, was later released for PS4 and Xbox One. 

I've also noted this information for future reference.
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing st

#### Advanced — UdaPlay as an explicit State Machine

Instead of letting the LLM decide which tool to call, wire the tools as fixed **nodes**
(Module 08 RAG style + Module 03 conditional routing):

`retrieve → evaluate → (web_search only if not useful) → generate`

In [10]:
# ADVANCED 2 — Convert the agent to an explicit state machine, with the tools as nodes.
# Reuses the same `collection`, `EvaluationReport`, LLM and Tavily client from above.
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Resource


class UdaPlayState(TypedDict):
    question: str
    retrieved_docs: List[Dict]
    evaluation: Optional[EvaluationReport]
    web_results: Optional[Dict]
    answer: str
    messages: List


def retrieve_node(state: UdaPlayState, resource: Resource) -> UdaPlayState:
    collection = resource.vars["collection"]
    results = collection.query(
        query_texts=[state["question"]],
        n_results=5,
        include=["metadatas"],
    )
    docs = [
        {
            "Platform": m["Platform"],
            "Name": m["Name"],
            "YearOfRelease": m["YearOfRelease"],
            "Description": m["Description"],
        }
        for m in results["metadatas"][0]
    ]
    return {"retrieved_docs": docs}


def evaluate_node(state: UdaPlayState, resource: Resource) -> UdaPlayState:
    llm = resource.vars["llm"]
    messages = [
        SystemMessage(content=(
            "Your task is to evaluate if the documents are enough to respond the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not."
        )),
        UserMessage(content=(
            f"# Question:\n{state['question']}\n\n"
            f"# Retrieved Documents:\n{state['retrieved_docs']}"
        )),
    ]
    ai_message = llm.invoke(input=messages, response_format=EvaluationReport)
    evaluation = PydanticOutputParser(model_class=EvaluationReport).parse(ai_message)
    return {"evaluation": evaluation}


def web_search_node(state: UdaPlayState, resource: Resource) -> UdaPlayState:
    tavily = resource.vars["tavily"]
    response = tavily.search(
        query=state["question"],
        search_depth="advanced",
        include_answer=True,
    )
    return {"web_results": {"answer": response.get("answer"), "results": response.get("results")}}


def generate_node(state: UdaPlayState, resource: Resource) -> UdaPlayState:
    llm = resource.vars["llm"]
    if state.get("web_results"):
        context = (
            f"Web answer: {state['web_results']['answer']}\n\n"
            f"Web results: {state['web_results']['results']}"
        )
        source = "a web search"
    else:
        context = "\n".join(str(doc) for doc in state["retrieved_docs"])
        source = "the internal game database"

    messages = [
        SystemMessage(content=(
            "You are UdaPlay, an AI research assistant for the video game industry. "
            "Answer the question using the provided context. "
            "If the context doesn't contain the answer, say you don't know."
        )),
        UserMessage(content=(
            f"# Question:\n{state['question']}\n\n"
            f"# Context (from {source}):\n{context}\n\n"
            "# Answer:"
        )),
    ]
    ai_message = llm.invoke(messages)
    return {"answer": ai_message.content, "messages": [ai_message]}

In [11]:
# Build the nodes
entry = EntryPoint()
retrieve_step = Step("retrieve", retrieve_node)
evaluate_step = Step("evaluate", evaluate_node)
web_search_step = Step("web_search", web_search_node)
generate_step = Step("generate", generate_node)
termination = Termination()


def route_after_evaluate(state: UdaPlayState):
    """If the retrieved docs are useful, answer directly; otherwise search the web first."""
    evaluation = state.get("evaluation")
    if evaluation and evaluation.useful:
        return generate_step
    return web_search_step


udaplay_machine = StateMachine(UdaPlayState)
udaplay_machine.add_steps(
    [entry, retrieve_step, evaluate_step, web_search_step, generate_step, termination]
)
udaplay_machine.connect(entry, retrieve_step)
udaplay_machine.connect(retrieve_step, evaluate_step)
udaplay_machine.connect(evaluate_step, [generate_step, web_search_step], route_after_evaluate)
udaplay_machine.connect(web_search_step, generate_step)
udaplay_machine.connect(generate_step, termination)

udaplay_resource = Resource(vars={
    "llm": LLM(model="gpt-4o-mini", temperature=0.0),
    "collection": collection,
    "tavily": TavilyClient(api_key=os.getenv("TAVILY_API_KEY")),
})

In [12]:
# Run the explicit UdaPlay state machine on the same test questions
for question in questions:
    run = udaplay_machine.run({"question": question}, udaplay_resource)
    print("=" * 80)
    print("Q:", question)
    print("A:", run.get_final_state()["answer"], "\n")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
Q: When was Pokémon Gold and Silver released?
A: Pokémon Gold and Silver were released in 1999. 

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
Q: Which one was the first 3D platformer Mario game?
A: The first 3D platformer Mario game is "Super Mario 64," released in 1996 for the Nintendo 64. 

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate
[StateMachine] Executing step: web_search
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
Q: Was Mortal Kombat X released for Playstation 5?
A: Mortal Kombat X was not originally released for PlayStati